# EDA — DonneesSportive.xlsx
## Sport Data Solution — Sport Data Analysis

**Purpose:** Understand the raw sport data structure, quality,
and content before building the activity generator.

**File:** `data/raw/DonneesSportive.xlsx`\
**Date:** 2026-03-13

## Section 1 - Setup & Imports

In [1]:
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 50)
warnings.filterwarnings('ignore')

DATA_PATH = Path("../data/raw/DonneesSportive.xlsx")

print("Librairies loaded successfully")
print(f"Data path: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Librairies loaded successfully
Data path: ../data/raw/DonneesSportive.xlsx
File exists: True


## Section 2 - Load Data
Loading the raw Excel file into a pandas DataFrame for analysis.

In [2]:
df = pd.read_excel(DATA_PATH, engine="openpyxl")

print("File loaded successfully")
print(f"Shape: {df.shape[0]} rows * {df.shape[1]} columns")

File loaded successfully
Shape: 161 rows * 2 columns


## Section 3 - Sample

Displaying the first 10 rows of raw data to confirm exact format of each column before designing ETL transformations.

In [3]:
print("=== FIRST 10 ROWS ===")
display(df.head(10))

=== FIRST 10 ROWS ===


,ID salarié,Pratique d'un sport
0,59019,NaN
1,19841,NaN
2,56482,Tennis
3,21886,NaN
4,81001,NaN
5,17757,Badminton
6,17036,Escalade
7,36913,NaN
8,79006,NaN
9,62296,NaN


## Section 4 - Shape & Column Names
Examining the exact structure and column naming of the dataset.

In [4]:
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\n=== Column Names ===")
for i, col in enumerate(df.columns):
    print(f" [{i}] {col}")

=== SHAPE ===
Rows: 161
Columns: 2

=== Column Names ===
 [0] ID salarié
 [1] Pratique d'un sport


### Observations

2 columns, 161 rows — matches exactly DonneesRH.xlsx row count.\
Suggests one sport declaration per employee.

Column naming issues identical to RH file:
- Accents, spaces, apostrophe → apply same snake_case cleaning in ETL.

## Section 5 - Data Types
Examining what pandas inferred as the data type for each column.\
Expected types are compared against inferred types to identify necessary transformations for the ETL piepeline. 

In [5]:
print("=== INFERRED DATA TYPES ===")
print(df.dtypes)

print("\n=== TYPE SUMMARY ===")
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"{dtype}: {count} columns")

=== INFERRED DATA TYPES ===
ID salarié             int64
Pratique d'un sport      str
dtype: object

=== TYPE SUMMARY ===
int64: 1 columns
str: 1 columns


### Observations

ID salarié: int64 -> must convert to string in ETL.\
Same reason as DonneesRH.xlsx, identifier not a number.\
Critical: must match rh_employee_id VARCHAR type for JOIN.

Pratique d'un sport: str - correct type, no conversion needed.

## Section 6 - Missing Values

Identifying columns with missing values, their count percentage.\
Missing values in NOT NULL columns will cause ETL failures and must be handle before loading into the database.

In [6]:
print("=== MISSING VALUES ===")
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) *100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
    'nullable_in_db': [False, False]
})

missing_df = missing_df.sort_values('missing_count', ascending=False)
print(missing_df)

print("\n=== SUMMARY ===")
total_missing = missing_count.sum()
columns_with_missing = (missing_count > 0).sum()
print(f"Total missing values: {total_missing}")
print(f"Columns with missing data: {columns_with_missing}/{df.shape[1]}")

=== MISSING VALUES ===
                     missing_count  missing_pct  nullable_in_db
Pratique d'un sport             66        40.99           False
ID salarié                       0         0.00           False

=== SUMMARY ===
Total missing values: 66
Columns with missing data: 1/2


### Observation

66/161 employees (40.99%) have no sport declared.\
Critical finding — impacts activity generation and well-being eligibility.

## Section 7 - Duplicates
Checking for two types of duplicates:
1. Full row duplicates: every column identical across rows
2. ID duplicates: same employee ID with different data

Both types will cause ETL failures due to PRIMARY KEY constraint on rh_employee_id in the empployees table.

In [7]:
print("=== FULL ROW DUPLICATES ===")
full_duplicates = df.duplicated()
print(f"Duplicate rows: {full_duplicates.sum()}")

if full_duplicates.sum() > 0:
    print("\nDuplicate rows content:")
    print(df[full_duplicates])

print("\n=== DUPLICATE EMPLOYEE ID ===")
id_column = 'ID salarié'
id_duplicates = df.duplicated(subset=[id_column])
print(f"Duplicate IDs: {id_duplicates.sum()}")

if id_duplicates.sum() > 0:
    duplicated_ids = df[df.duplicated(subset=[id_column], keep=False)]
    print("\nRows with duplicate IDs:")
    print(duplicated_ids.sort_values(id_column))

print("\n=== SUMMARY ===")
print(f"Total rows: {len(df)}")
print(f"Full rows duplicates: {full_duplicates.sum()}")
print(f"Duplicate IDs: {id_duplicates.sum()}")
print(f"Unique employee IDs: {df[id_column].nunique()}")

=== FULL ROW DUPLICATES ===
Duplicate rows: 0

=== DUPLICATE EMPLOYEE ID ===
Duplicate IDs: 0

=== SUMMARY ===
Total rows: 161
Full rows duplicates: 0
Duplicate IDs: 0
Unique employee IDs: 161


### Observation

No duplicates found. 161 unique employee IDs confirmed.

## Section 8 - Unique values
Analysing unique values for categorical columns.\
Critical for validating ENUM mappings and transport mode values that must match our database ENUM type definitions

In [8]:
CATEGORICAL_THRESHOLD = 15

print("=== UNIQUE VALUES PER COLUMN ===\n")
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"--- {col} ({n_unique} unique values) ---")
    
    if n_unique <= CATEGORICAL_THRESHOLD:
        value_counts = df[col].value_counts()
        for value, count in value_counts.items():
            pct = round(count / len(df) * 100, 2)
            print(f"  {value:<40} count: {count:<5} ({pct}%)")
    else:
        print(f"  High cardinality — {n_unique} unique values, sample:")
        for val in df[col].head(3):
            print(f"  {val}")
    print()

=== UNIQUE VALUES PER COLUMN ===

--- ID salarié (161 unique values) ---
  High cardinality — 161 unique values, sample:
  59019
  19841
  56482

--- Pratique d'un sport (15 unique values) ---
  Runing                                   count: 18    (11.18%)
  Randonnée                                count: 16    (9.94%)
  Tennis                                   count: 11    (6.83%)
  Natation                                 count: 8     (4.97%)
  Football                                 count: 6     (3.73%)
  Rugby                                    count: 6     (3.73%)
  Badminton                                count: 5     (3.11%)
  Voile                                    count: 5     (3.11%)
  Judo                                     count: 4     (2.48%)
  Boxe                                     count: 4     (2.48%)
  Escalade                                 count: 3     (1.86%)
  Triathlon                                count: 3     (1.86%)
  Équitation                          

### Observation

15 unique sport values found. Typo detected: "Runing" -> "Running".\
Most sports map to 'other' ENUM value.

## Section 9 - Statistics
Statistical analysis of numeric, categorical and date columns.\
Outliers and anomalies identified here must be handle in ETL.

In [9]:
print("=== NUMERIC COLUMNS ===")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(df[numeric_cols].describe().round(2))

print("\n=== CATEGORICAL COLUMNS ===")
categorical_cols = df.select_dtypes(include=['object', 'str']).columns
for col in categorical_cols:
    if df[col].nunique() <= CATEGORICAL_THRESHOLD:
        print(f"\n{col}:")
        print(f"Most frequent: {df[col].mode()[0]} ({df[col].value_counts().iloc[0]} occurrences)")
        print(f"Least frequent: {df[col].value_counts().index[-1]} ({df[col].value_counts().iloc[-1]} occurrences)")

=== NUMERIC COLUMNS ===
       ID salarié
count      161.00
mean     52622.02
std      24202.83
min      12497.00
25%      33386.00
50%      49804.00
75%      71657.00
max      99514.00

=== CATEGORICAL COLUMNS ===

Pratique d'un sport:
Most frequent: Runing (18 occurrences)
Least frequent: Basketball (2 occurrences)


### Observation 

ID statistics identical to DonneesRH.xlsx — same employee population.\
Most frequent sport: Running (18). Least frequent: Basketball (2).

## Section 10 - Deep Address Analysis

Four targeted checks critical for activity generator and database loading.


In [12]:
df_rh = pd.read_excel(Path("../data/raw/DonneesRH.xlsx"))
df_rh['ID salarié'] = df_rh['ID salarié'].astype(str)
df['ID salarié'] = df['ID salarié'].astype(str)

SPORT_MAPPING = {
    'runing': 'running',
    'running': 'running',
    'triathlon': 'running',
    'randonnée': 'hiking',
    'natation': 'swimming',
    'tennis': 'racket_sports',
    'badminton': 'racket_sports',
    'tennis de table': 'racket_sports',
    'boxe': 'combat_sports',
    'judo': 'combat_sports',
    'football': 'team_sports',
    'rugby': 'team_sports',
    'basketball': 'team_sports',
    'voile': 'outdoor_sports',
    'équitation': 'outdoor_sports',
    'escalade': 'outdoor_sports',
}

print("=== CHECK 1: CASE CONSISTENCY ===")
sports_with_nulls = df['Pratique d\'un sport'].dropna()
case_issues = []
for sport in sports_with_nulls:
    normalized = sport.lower().strip()
    if sport != sport.strip():
        case_issues.append((sport, 'trailing/leading spaces'))
    if sport != sport and normalized in SPORT_MAPPING:
        case_issues.append((sport, 'case mismatch'))

unique_raw = sports_with_nulls.unique()
print(f"Raw unique values ({len(unique_raw)}):")
for val in sorted(unique_raw):
    normalized = val.lower().strip()
    mapped = SPORT_MAPPING.get(normalized, 'UNMAPPED')
    spaces = '  [trailing space]' if val != val.strip() else ''
    print(f"  '{val}'{spaces} -> {mapped}")

print("\n=== CHECK 2: MULTIPLE SPORTS PER EMPLOYEE ===")
id_counts = df.groupby('ID salarié')['Pratique d\'un sport'].count()
multi_sport = id_counts[id_counts > 1]
print(f"Employees with multiple sport rows: {len(multi_sport)}")
if len(multi_sport) > 0:
    print(multi_sport)
else:
    print("  One row per employee confirmed")

print("\n=== CHECK 3: CROSS-REFERENCE WITH RH DATA ===")
sport_ids = set(df['ID salarié'].unique())
rh_ids = set(df_rh['ID salarié'].astype(str).unique())

in_sport_not_rh = sport_ids - rh_ids
in_rh_not_sport = rh_ids - sport_ids

print(f"IDs in sport file but NOT in RH file: {len(in_sport_not_rh)}")
if in_sport_not_rh:
    print(f"  Orphan IDs: {in_sport_not_rh}")

print(f"IDs in RH file but NOT in sport file: {len(in_rh_not_sport)}")
if in_rh_not_sport:
    print(f"  Missing IDs: {list(in_rh_not_sport)[:5]} ...")

print(f"IDs present in both files: {len(sport_ids & rh_ids)}")

print("\n=== CHECK 4: ENUM MAPPING VALIDATION ===")
unmapped = []
for sport in sports_with_nulls:
    normalized = sport.lower().strip()
    if normalized not in SPORT_MAPPING:
        unmapped.append(sport)

print(f"Unmapped sports: {len(unmapped)}")
if unmapped:
    for s in set(unmapped):
        print(f"  '{s}' → no mapping defined")
else:
    print("  All sports successfully mapped to ENUM categories")

print("\n=== MAPPING SUMMARY ===")
mapped_counts = {}
for sport in sports_with_nulls:
    normalized = sport.lower().strip()
    category = SPORT_MAPPING.get(normalized, 'other')
    mapped_counts[category] = mapped_counts.get(category, 0) + 1

for category, count in sorted(mapped_counts.items(),
                               key=lambda x: x[1], reverse=True):
    pct = f"{count / len(sports_with_nulls) * 100:.1f}"
    print(f"  {category:<20} count: {count:<5} ({pct}%)")

=== CHECK 1: CASE CONSISTENCY ===
Raw unique values (15):
  'Badminton' -> racket_sports
  'Basketball' -> team_sports
  'Boxe' -> combat_sports
  'Escalade' -> outdoor_sports
  'Football' -> team_sports
  'Judo' -> combat_sports
  'Natation' -> swimming
  'Randonnée' -> hiking
  'Rugby' -> team_sports
  'Runing' -> running
  'Tennis' -> racket_sports
  'Tennis de table' -> racket_sports
  'Triathlon' -> running
  'Voile' -> outdoor_sports
  'Équitation' -> outdoor_sports

=== CHECK 2: MULTIPLE SPORTS PER EMPLOYEE ===
Employees with multiple sport rows: 0
  One row per employee confirmed

=== CHECK 3: CROSS-REFERENCE WITH RH DATA ===
IDs in sport file but NOT in RH file: 0
IDs in RH file but NOT in sport file: 0
IDs present in both files: 161

=== CHECK 4: ENUM MAPPING VALIDATION ===
Unmapped sports: 0
  All sports successfully mapped to ENUM categories

=== MAPPING SUMMARY ===
  running              count: 21    (22.1%)
  racket_sports        count: 18    (18.9%)
  hiking             

### Observation

All 4 checks passed. Zero issues found.\
15 sports fully mapped to 7 ENUM categories.\
161 IDs present in both files — zero FK risk.\
One row per employee confirmed — no duplicate risk.

## Section 11 — Final Conclusions

### Dataset Overview
- Source: DonneesSportive.xlsx
- Rows: 161 employees
- Columns: 2
- Cross-reference: 161/161 IDs match DonneesRH.xlsx

---

### Data Quality Summary

| Dimension             | Result          | Action needed      |
|-----------------------|-----------------|--------------------|
| Missing values        | 66/161 (40.99%) | Skip in generator  |
| Full duplicates       | 0               | None               |
| Duplicate IDs         | 0               | None               |
| Unmapped sports       | 0               | None               |
| Cross-reference       | 161/161 match   | None               |
| Case inconsistencies  | 1 (Runing)      | Handled in mapping |

---

### ETL Actions Required

**1. Type conversion**
ID salarié: int64 -> string before any JOIN operation with employees table.

**2. Sport mapping**
Apply SPORT_MAPPING dictionary with .lower().strip() normalization before inserting into sport_activities.\
Handles typo "Runing" → "running" automatically.

**3. Handle missing sport declarations**
66 employees have NULL sport value.\
Generator must skip these employees entirely.\
They remain in employees table and are still eligible for prime benefit — sport practice does not affect prime eligibility.

**4. ENUM expansion required**
Current activity_type ENUM is insufficient.\
V4 migration must expand to 9 categories:\
  running, walking, cycling, hiking, swimming,\
  racket_sports, combat_sports, team_sports,\
  outdoor_sports, other

---

### Sport Distribution (95 employees with declared sport)

| Category       | Count | Percentage |
|----------------|-------|------------|
| running        | 21    | 22.1%      |
| racket_sports  | 18    | 18.9%      |
| hiking         | 16    | 16.8%      |
| team_sports    | 14    | 14.7%      |
| outdoor_sports | 10    | 10.5%      |
| swimming       | 8     | 8.4%       |
| combat_sports  | 8     | 8.4%       |

---

### Impact on Well-being Days Eligibility
```
Total employees: 161
With sport declared: 95 (59.0%)
Without sport declared: 66 (41.0%)

Maximum potentially eligible
for well-being days: 95
(still requires minimum 15 activities in the year)
```

### Key Decision — Activity Generator Scope

Generator will only create activities for 95 employees.\
66 employees with NULL sport are excluded from generation.\
This is a business decision — not a technical limitation.\
If these employees declare a sport in future:\
-> Kestra re-runs generator for them automatically\
-> historical periods remain at 0 activities

---

### Confidence Level
Dataset is clean and consistent with DonneesRH.xlsx.\
Zero cross-reference issues — no FK risk.\
One typo handled in mapping.\
66 missing values documented and handled.\
Generator implementation can proceed.